# Task 5 -- Three-Way Evaluation and Verdict
### BITS Pilani WILP M.Tech. AIML -- Conversational AI (AIMLCZG521) -- PS2

| Field | Value |
|---|---|
| **Notebook** | 05 of 05 -- Task 5: Three-Way Evaluation and Verdict (2 marks) |
| **Systems compared** | (A) base SLM &nbsp;\|&nbsp; (B) reference LLM &nbsp;\|&nbsp; (C) fine-tuned SLM v2 |

**MUST RUN IN THE BITS VIRTUAL LAB** (loads all three models). This notebook was authored and
syntax-checked outside the Virtual Lab; **all scores, metrics and error analysis must be produced by
actually executing it in the Virtual Lab**, per PS2's submission rules.

**Objective:** answer PS2's central question with evidence -- *how close did the fine-tuned SLM (C)
get to the large reference LLM (B)?* -- via a frozen-benchmark side-by-side, a disclosed rubric (or
LLM-as-judge), an automatic metric on the held-out test set, honest error analysis, and a deployment
verdict.

## Imports and Environment Setup

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch

from src.config import (SLM_MODEL_ID, REFERENCE_LLM_MODEL_ID, SEED, PROCESSED_DIR,
                          ADAPTER_V2_DIR, RESULTS_DIR, GENERATION_KWARGS, SYSTEM_PROMPT, BENCHMARK_PATH)
from src.data_utils import load_split, load_benchmark, format_chat_prompt
from src.model_utils import load_causal_lm, load_lora_adapter, generate_response
from src.eval_utils import RUBRIC_CRITERIA, RUBRIC_SCALE, new_score_sheet, score_totals, rouge_l_f1, save_json

np.random.seed(SEED)
torch.manual_seed(SEED)
benchmark = load_benchmark(BENCHMARK_PATH)
test_records = load_split(PROCESSED_DIR, "test")
print(f"Frozen benchmark: {len(benchmark)} prompts | Held-out test set: {len(test_records)} examples")


## Step 1 -- Side-by-Side Comparison on the Frozen Benchmark (A / B / C)

Reuses Notebook 02's saved (A) and (B) responses where available (identical model + identical
generation settings -> no need to regenerate), and generates fresh (C) responses from the v2 adapter.

In [ ]:
baseline_path = os.path.join(RESULTS_DIR, "task2_baseline_responses.json")
with open(baseline_path) as f:
    baseline = json.load(f)
responses_A = baseline["system_A_base_slm"]["responses"]
responses_B = baseline["system_B_reference_llm"]["responses"]

base_model_c, tokenizer_c = load_causal_lm(SLM_MODEL_ID, quantize_4bit=True)
model_C = load_lora_adapter(base_model_c, ADAPTER_V2_DIR)
model_C.eval()

responses_C = {}
for item in benchmark:
    prompt_text = format_chat_prompt(tokenizer_c, item["prompt"], SYSTEM_PROMPT)
    responses_C[item["id"]] = generate_response(model_C, tokenizer_c, prompt_text, **GENERATION_KWARGS)

rows = []
for item in benchmark:
    rows.append({
        "id": item["id"], "type": item["type"], "prompt": item["prompt"],
        "A_base_slm": responses_A[item["id"]],
        "B_reference_llm": responses_B[item["id"]],
        "C_finetuned_slm_v2": responses_C[item["id"]],
    })
three_way = pd.DataFrame(rows)
three_way.to_csv(os.path.join(RESULTS_DIR, "task5_three_way_comparison.csv"), index=False)
for _, row in three_way.iterrows():
    print(f"=== [{row['id']}] ({row['type']}) ===")
    print("Q:", row["prompt"])
    print("A:", row["A_base_slm"])
    print("B:", row["B_reference_llm"])
    print("C:", row["C_finetuned_slm_v2"])
    print()


## Step 2 -- Scored Evaluation (disclosed manual rubric)

**Rubric (0-2 per criterion (0=fails, 1=partially meets, 2=fully meets)):**
- `domain_correctness` -- is the banking information factually right for NovaBank's stated policies/process?
- `instruction_format_adherence` -- did the response follow any explicit formatting instruction (esp. B4)?
- `completeness` -- does it fully answer, including every sub-part (esp. the 2-step B5 reasoning)?
- `hallucination` -- **inverted scale**: 2 = no fabricated facts/policies, 0 = clearly fabricated

Score each system x each of the 8 prompts by hand after reading Step 1's real outputs (do not score
before reading -- PS2 penalises fabricated/cherry-picked results). An **LLM-as-judge alternative** with
the exact judging prompt disclosed is provided in the next cell if you prefer an automated first pass
(still spot-check a few by hand).

In [ ]:
score_A = new_score_sheet("A_base_slm", benchmark)
score_B = new_score_sheet("B_reference_llm", benchmark)
score_C = new_score_sheet("C_finetuned_slm_v2", benchmark)

for df, resp in [(score_A, responses_A), (score_B, responses_B), (score_C, responses_C)]:
    for pid in df.index:
        df.loc[pid, "response"] = resp[pid]

# --- MANUALLY FILL IN the 4 rubric columns for each of the 3 sheets below, e.g.: ---
# score_A.loc["B1", "domain_correctness"] = 2
# score_A.loc["B1", "instruction_format_adherence"] = 2
# score_A.loc["B1", "completeness"] = 1
# score_A.loc["B1", "hallucination"] = 2
# ... (repeat for all 8 prompt ids x 3 systems)

score_A.to_csv(os.path.join(RESULTS_DIR, "task5_rubric_A.csv"))
score_B.to_csv(os.path.join(RESULTS_DIR, "task5_rubric_B.csv"))
score_C.to_csv(os.path.join(RESULTS_DIR, "task5_rubric_C.csv"))
score_A


### Optional: LLM-as-judge (disclosed prompt) using the reference LLM (B) as judge

If you use this path, disclose it explicitly in the report (as done here) since B is also one of the
systems being judged -- a known potential bias (a model preferring its own style). Spot-check a handful
of the judge's scores by hand to sanity-check this.

In [ ]:
JUDGE_PROMPT_TEMPLATE = """You are an impartial evaluator of a banking customer-support assistant's response.

User question: {prompt}

Assistant response to evaluate:
{response}

Score the response from 0-2 on each criterion (0=fails, 1=partially meets, 2=fully meets):
- domain_correctness: is the banking information factually right?
- instruction_format_adherence: did it follow any explicit formatting instruction in the question?
- completeness: does it fully answer, without missing steps?
- hallucination (inverted): 2 = no fabricated facts, 0 = fabricated facts present

Respond ONLY with a JSON object like:
{{"domain_correctness": 0, "instruction_format_adherence": 0, "completeness": 0, "hallucination": 0}}
"""

def llm_judge_score(judge_model, judge_tokenizer, prompt, response):
    """
    Call the reference LLM as an automated judge for one (prompt, response) pair,
    using the disclosed JUDGE_PROMPT_TEMPLATE above. Returns a dict of the 4
    rubric scores parsed from the judge's JSON reply, or None if parsing fails
    (fall back to manual scoring for that cell).
    """
    judge_input = JUDGE_PROMPT_TEMPLATE.format(prompt=prompt, response=response)
    judge_chat = format_chat_prompt(judge_tokenizer, judge_input, "You are a strict, impartial evaluator.")
    raw = generate_response(judge_model, judge_tokenizer, judge_chat, max_new_tokens=120, do_sample=False, repetition_penalty=1.0)
    try:
        start, end = raw.index("{"), raw.rindex("}") + 1
        return json.loads(raw[start:end])
    except (ValueError, json.JSONDecodeError):
        return None

# Example usage (uncomment to run against the already-loaded reference LLM if still in memory,
# or reload it as in Notebook 02):
# base_model_b, tokenizer_b = load_causal_lm(REFERENCE_LLM_MODEL_ID, quantize_4bit=True)
# judge_result = llm_judge_score(base_model_b, tokenizer_b, benchmark[0]["prompt"], responses_C[benchmark[0]["id"]])
# print(judge_result)


## Step 3 -- Per-System Totals

In [ ]:
totals = {
    "A_base_slm": score_totals(score_A),
    "B_reference_llm": score_totals(score_B),
    "C_finetuned_slm_v2": score_totals(score_C),
}
totals_df = pd.DataFrame(totals).T
totals_df["grand_total"] = totals_df.sum(axis=1)
totals_df.to_csv(os.path.join(RESULTS_DIR, "task5_rubric_totals.csv"))
totals_df


## Step 4 -- Automatic Metric on the Held-Out Test Set (A vs. C)

**Metric: ROUGE-L F1** (longest-common-subsequence overlap) between each system's generated response
and the ground-truth `output` for every held-out test example, comparing (A) base SLM vs. (C)
fine-tuned v2 -- (B) is excluded here since the reference LLM was never trained to imitate this
dataset's specific response style, so a style-overlap metric would unfairly penalise a *correct but
differently-phrased* answer from B.

**Limitation (state in the report):** ROUGE-L measures *surface lexical/sequence overlap*, not semantic
correctness -- a response can be factually right but phrased differently from the reference (scoring
low) or can copy reference phrasing while being subtly wrong (scoring high). It should be read alongside
the rubric scores above, not in isolation. If `bert-score` is installed, a semantic-similarity metric is
also computed below as a partial mitigation (still not a full substitute for the manual rubric).

In [ ]:
base_model_a, tokenizer_a = load_causal_lm(SLM_MODEL_ID, quantize_4bit=False)

rouge_A, rouge_C = [], []
for r in test_records:
    prompt_text_a = format_chat_prompt(tokenizer_a, r["instruction"], SYSTEM_PROMPT)
    gen_a = generate_response(base_model_a, tokenizer_a, prompt_text_a, **GENERATION_KWARGS)
    rouge_A.append(rouge_l_f1(gen_a, r["output"]))

    prompt_text_c = format_chat_prompt(tokenizer_c, r["instruction"], SYSTEM_PROMPT)
    gen_c = generate_response(model_C, tokenizer_c, prompt_text_c, **GENERATION_KWARGS)
    rouge_C.append(rouge_l_f1(gen_c, r["output"]))

metric_summary = {
    "metric": "ROUGE-L F1",
    "n_test_examples": len(test_records),
    "A_base_slm_mean": float(np.mean(rouge_A)),
    "C_finetuned_slm_v2_mean": float(np.mean(rouge_C)),
    "delta_C_minus_A": float(np.mean(rouge_C) - np.mean(rouge_A)),
}
save_json(metric_summary, os.path.join(RESULTS_DIR, "task5_automatic_metric.json"))
print(json.dumps(metric_summary, indent=2))


In [ ]:
# Optional semantic-similarity metric (only runs if bert-score is installed):
try:
    from bert_score import score as bertscore
    _, _, f1_A = bertscore([r["output"] for r in test_records], [r["output"] for r in test_records], lang="en", verbose=False)
    print("bert-score available -- consider adding BERTScore F1 for A vs C alongside ROUGE-L (see report note).")
except ImportError:
    print("bert-score not installed; ROUGE-L above is the primary automatic metric (as documented in its limitation note).")


## Step 5 -- Honest Error Analysis

**Required: at least two cases where the fine-tuned SLM (C) still loses to the reference LLM (B)**, each
with a one-paragraph diagnosis of whether the remaining gap is a **data problem** (the 338-example
NovaBank set doesn't cover this case), a **capacity problem** (1.5B parameters cannot represent the
required reasoning/knowledge even with perfect data), or a **training problem** (fixable with more
epochs / different hyperparameters / a different PEFT config).

Pick 2 concrete prompt ids from Step 1's three-way table where B clearly outperforms C (candidates to
check first: **B5**, the multi-step reasoning prompt -- small models often struggle with arithmetic
reasoning regardless of fine-tuning data, which would point to a *capacity* problem; and **B2**, the
loan-documents factual prompt, if C's answer is thinner than B's -- more plausibly a *data* problem
given only ~13 training examples for `apply_for_loan`).

**Case 1 -- prompt *(fill in id)*:**
C's response: *(paste)*
B's response: *(paste)*
Diagnosis (data / capacity / training, with reasoning): *(fill in, 1 paragraph)*

**Case 2 -- prompt *(fill in id)*:**
C's response: *(paste)*
B's response: *(paste)*
Diagnosis (data / capacity / training, with reasoning): *(fill in, 1 paragraph)*

## Step 6 -- Verdict

**Fill in after reviewing Steps 1-5's real results.** Address, in one paragraph: given the measured
quality gap (rubric totals + ROUGE-L delta), NovaBank's cost/privacy/hardware constraints (Task 3's
efficiency numbers -- trainable-parameter % and peak training memory), and the error analysis above --
**when would you deploy the fine-tuned SLM (C) instead of the reference LLM (B), and when not?**
Consider: is C "close enough" for high-volume, template-like intents (e.g. card activation, password
reset) even if it still trails B on open-ended reasoning (B5) or thin-coverage intents? Would a hybrid
(SLM handles routine intents, escalates low-confidence/reasoning-heavy cases to the LLM) be a more
honest recommendation than an all-or-nothing choice?

*(Write the verdict paragraph here.)*

## Conclusion

- Produced the three-way (A/B/C) side-by-side on the frozen 8-prompt benchmark.
- Scored all three systems on a disclosed 4-criterion, 0-2 rubric (with an optional disclosed
  LLM-as-judge path), reporting per-system totals.
- Computed ROUGE-L F1 (A vs. C) on the full held-out test set, with its surface-overlap limitation
  stated explicitly.
- Documented >=2 honest error cases where C still loses to B, each diagnosed as a data / capacity /
  training problem.
- Delivered a verdict on when the fine-tuned SLM is deployment-ready vs. when the reference LLM (or a
  hybrid) is still required.

This closes PS2's central question: how close a parameter-efficiently fine-tuned 1.5B SLM, trained on
~300 domain examples, can get to a ~7B reference LLM on in-domain banking-support tasks -- with the
gap that remains reported honestly rather than hidden.